# Car Damage Severity Detection with Vertex AI

## Project Overview

This project demonstrates a complete, cloud-based computer vision workflow using Google Cloud's Vertex AI platform. The goal is to build, train, and deploy an AutoML image classification model capable of detecting the severity of car damage from an image.

The project covers the entire lifecycle:
1.  **Setup & Authentication:** Configuring the local environment to communicate with Google Cloud.
2.  **Data Preparation:** Preparing and uploading a custom image dataset to Google Cloud Storage.
3.  **Model Training:** Using Vertex AI AutoML to automatically train a state-of-the-art vision model.
4.  **Deployment & Prediction:** Deploying the trained model to a live endpoint and making a real-time prediction.
5.  **Cleanup:** Deleting all cloud resources to manage costs.

--- 
## 1. Setup and Library Imports

In [1]:
from google.cloud import storage
from google.cloud import aiplatform
import base64
import time
import os
import pandas as pd

### 1.1. Google Cloud Authentication

The following command authenticates the notebook environment, allowing it to securely interact with Google Cloud services. This only needs to be run once per session.

In [3]:
!gcloud auth application-default login

Your browser has been opened to visit:

    https://accounts.google.com/o/oauth2/auth?response_type=code&client_id=764086051850-6qr4p6gpi6hn506pt8ejuq83di341hur.apps.googleusercontent.com&redirect_uri=http%3A%2F%2Flocalhost%3A8085%2F&scope=openid+https%3A%2F%2Fwww.googleapis.com%2Fauth%2Fuserinfo.email+https%3A%2F%2Fwww.googleapis.com%2Fauth%2Fcloud-platform+https%3A%2F%2Fwww.googleapis.com%2Fauth%2Fsqlservice.login&state=zszXatasTaJyAw5LZR1tJlZhnfOmY9&access_type=offline&code_challenge=onZgLZ0C-39vPGnzjfh9-CgqN6_CTMsbCs7kMHGF5hw&code_challenge_method=S256


Credentials saved to file: [C:\Users\redae\AppData\Roaming\gcloud\application_default_credentials.json]

These credentials will be used by any library that requests Application Default Credentials (ADC).

Quota project "reda-car-damage-vision" was added to ADC which can be used by Google client libraries for billing and quota. Note that some services may still bill the project owning the resource.


--- 
## 2. Data Preparation and Upload

This section handles the preparation of the image dataset. The process involves:
1. Creating a dedicated bucket in Google Cloud Storage.
2. Uploading the entire local image dataset to the new bucket, preserving the folder structure (`training/minor`, `training/moderate`, etc.).

In [13]:
# Define project-specific configuration variables.
PROJECT_ID = "reda-car-damage-vision"
BUCKET_NAME = f"car-damage-dataset-{PROJECT_ID}"
LOCAL_DATA_PATH = "C:/Users/redae/Jupyter/Projects/Car Damage Vision/Car Damage Severity Dataset"

# Create a new Cloud Storage bucket if it doesn't already exist.
storage_client = storage.Client(project=PROJECT_ID)
try:
    print(f"Creating a new Cloud Storage bucket: {BUCKET_NAME}")
    bucket = storage_client.create_bucket(BUCKET_NAME, location="us-central1")
    print("Bucket created successfully.")
except Exception as e:
    print(f"Bucket creation failed: {e}. It likely already exists. Using existing bucket.")
    bucket = storage_client.get_bucket(BUCKET_NAME)

# Define a function to upload all files while preserving the directory structure.
def upload_dataset_to_gcs(local_path, gcs_bucket):
    """Uploads all files from a local directory to a GCS bucket."""
    print("\nStarting dataset upload (this may take a few minutes)...")
    for local_folder, _, files in os.walk(local_path):
        for file_name in files:
            local_file_path = os.path.join(local_folder, file_name)
            relative_path = os.path.relpath(local_file_path, local_path)
            gcs_file_path = relative_path.replace("\\", "/")
            blob = gcs_bucket.blob(gcs_file_path)
            blob.upload_from_filename(local_file_path)
    print(f"Finished uploading files from {local_path}")

# Call the function to upload the data.
upload_dataset_to_gcs(LOCAL_DATA_PATH, bucket)
print("\nUpload complete.")

Creating a new Cloud Storage bucket: car-damage-dataset-reda-car-damage-vision
Bucket creation failed: 409 POST https://storage.googleapis.com/storage/v1/b?project=reda-car-damage-vision&prettyPrint=false: Your previous request to create the named bucket succeeded and you already own it.. It likely already exists. Using existing bucket.

Starting dataset upload (this may take a few minutes)...
Finished uploading files from C:/Users/redae/Jupyter/Projects/Car Damage Vision/Car Damage Severity Dataset

Upload complete.


### 2.1. Create a CSV Index File

To ensure Vertex AI can reliably find and label all images, we will create a CSV file that maps each image's cloud storage path to its correct label (e.g., 'minor', 'moderate', 'severe'). This is a robust alternative to letting the service infer labels from folder names.

In [15]:
# Define project configuration.
PROJECT_ID = "reda-car-damage-vision"
BUCKET_NAME = f"car-damage-dataset-{PROJECT_ID}"
LOCAL_DATA_PATH = "C:/Users/redae/Jupyter/Projects/Car Damage Vision/Car Damage Severity Dataset"

# Automate the creation of the CSV index file.
image_paths = []
labels = []

# Walk through the local training directory to gather file paths and labels.
training_path = os.path.join(LOCAL_DATA_PATH, 'training')
for dirpath, _, filenames in os.walk(training_path):
    for filename in filenames:
        if filename.endswith(('.jpg', '.jpeg', '.png')):
            gcs_path = f"gs://{BUCKET_NAME}/training/{os.path.basename(dirpath)}/{filename}"
            image_paths.append(gcs_path)
            labels.append(os.path.basename(dirpath))

# Create and save a pandas DataFrame to a CSV file.
dataset_df = pd.DataFrame({
    'gcs_path': image_paths,
    'label': labels
})
output_csv_path = 'dataset.csv'
dataset_df.to_csv(output_csv_path, index=False, header=False)

print(f"Successfully created '{output_csv_path}' with {len(dataset_df)} entries.")

Successfully created 'dataset.csv' with 460 entries.


--- 
## 3. Vertex AI Dataset Creation and Model Training

With the data in the cloud, we can now create a managed Vertex AI Dataset and launch the AutoML training job.

### 3.1. Create Vertex AI Dataset

This step points Vertex AI to the `dataset.csv` file in our Cloud Storage bucket, creating a managed dataset resource that the platform can use for training.

In [7]:
# Define configuration and initialize the Vertex AI SDK.
PROJECT_ID = "reda-car-damage-vision"
BUCKET_URI = f"gs://car-damage-dataset-{PROJECT_ID}"
DATASET_DISPLAY_NAME = "car-damage-dataset"
aiplatform.init(project=PROJECT_ID, location="us-central1", staging_bucket=BUCKET_URI)

# Create the Vertex AI Dataset from the CSV file for reliability.
print(f"Creating Vertex AI Dataset from {BUCKET_URI}/dataset.csv...")
dataset = aiplatform.ImageDataset.create(
    display_name=DATASET_DISPLAY_NAME,
    gcs_source=[f"{BUCKET_URI}/dataset.csv"],
    import_schema_uri=aiplatform.schema.dataset.ioformat.image.single_label_classification,
)

print(f"Dataset created successfully. Resource name: {dataset.resource_name}")

Creating Vertex AI Dataset from gs://car-damage-dataset-reda-car-damage-vision/dataset.csv...
Creating ImageDataset
Create ImageDataset backing LRO: projects/601447762723/locations/us-central1/datasets/6821512525478100992/operations/4243659101285908480
ImageDataset created. Resource name: projects/601447762723/locations/us-central1/datasets/6821512525478100992
To use this ImageDataset in another session:
ds = aiplatform.ImageDataset('projects/601447762723/locations/us-central1/datasets/6821512525478100992')
Importing ImageDataset data: projects/601447762723/locations/us-central1/datasets/6821512525478100992
Import ImageDataset data backing LRO: projects/601447762723/locations/us-central1/datasets/6821512525478100992/operations/5608249788379168768
ImageDataset data imported. Resource name: projects/601447762723/locations/us-central1/datasets/6821512525478100992
Dataset created successfully. Resource name: projects/601447762723/locations/us-central1/datasets/6821512525478100992


### 3.2. Train AutoML Model

This cell kicks off the AutoML training job. Vertex AI will automatically search for the best model architecture for this specific dataset. This process is computationally intensive and will take several hours to complete. The `sync=False` parameter allows the script to finish immediately while the job runs in the cloud.

In [21]:
# Define the model display name.
MODEL_DISPLAY_NAME = "car_damage_severity_classifier"

# Create and configure the training job.
job = aiplatform.AutoMLImageTrainingJob(
    display_name = f"{MODEL_DISPLAY_NAME}_training_job",
    prediction_type = "classification",
    model_type = "CLOUD",
    base_model = None,
)

# Run the training job.
model = job.run(
    dataset = dataset,
    model_display_name = MODEL_DISPLAY_NAME,
    training_fraction_split = 0.8,
    validation_fraction_split = 0.1,
    test_fraction_split = 0.1,
    budget_milli_node_hours = 8000, # 8 node hours is the minimum for this model type
    sync = False, # Run asynchronously in the cloud
)

print("\nTraining job started. This will take several hours to complete.")
print("You can monitor the progress in the Google Cloud Console under 'Vertex AI' > 'Training'.")
print(f"To see your model after training, go to 'Vertex AI' > 'Models'.")

Starting AutoML model training for: car_damage_severity_classifier...

Training job started. This will take several hours to complete.
You can monitor the progress in the Google Cloud Console under 'Vertex AI' > 'Training'.
To see your model after training, go to 'Vertex AI' > 'Models'.
View Training:
https://console.cloud.google.com/ai/platform/locations/us-central1/training/1626392737903280128?project=601447762723
AutoMLImageTrainingJob projects/601447762723/locations/us-central1/trainingPipelines/1626392737903280128 current state:
2


--- 
## 4. Deployment and Prediction

Once the model has finished training, this section reloads the necessary resources from the cloud, deploys the model to a live endpoint, and makes a prediction on a new, unseen image.

### 4.1. Setup for a New Session

This cell is used to re-initialize the notebook environment and get references to the existing cloud resources (model, dataset, bucket) without having to re-run the entire pipeline. It finds resources by their unique IDs for reliability.

In [13]:
# Define all necessary configuration variables and unique IDs.
PROJECT_ID = "reda-car-damage-vision"
LOCATION = "us-central1"
MODEL_ID = "576094614931374080"
BUCKET_NAME = f"car-damage-dataset-{PROJECT_ID}"
DATASET_DISPLAY_NAME = "car-damage-dataset"

# Initialize the Vertex AI SDK.
aiplatform.init(project=PROJECT_ID, location=LOCATION)

# Get a direct reference to the trained model using its unique ID.
print("Finding existing model directly by ID...")
try:
    model_resource_name = f"projects/{PROJECT_ID}/locations/{LOCATION}/models/{MODEL_ID}"
    model = aiplatform.Model(model_name=model_resource_name)
    print(f"Found model: {model.resource_name}")
except Exception as e:
    print(f"Could not find model directly. Error: {e}")

# Get a reference to the dataset.
print("\nFinding existing dataset...")
datasets = aiplatform.ImageDataset.list(filter=f"display_name='{DATASET_DISPLAY_NAME}'")
if datasets:
    dataset = datasets[0]
    print(f"Found dataset: {dataset.resource_name}")
else:
    print("Dataset not found.")

# Get a reference to the bucket.
print("\nFinding existing GCS bucket...")
try:
    storage_client = storage.Client(project=PROJECT_ID)
    bucket = storage_client.get_bucket(BUCKET_NAME)
    print(f"Found bucket: {bucket.name}")
except Exception as e:
    print(f"Could not find bucket: {e}")

print("\nSetup complete. You can now run the deployment cell.")

Finding existing model directly by ID...
Found model: projects/601447762723/locations/us-central1/models/576094614931374080

Finding existing dataset...
Dataset not found.

Finding existing GCS bucket...
Found bucket: car-damage-dataset-reda-car-damage-vision

Setup complete. You can now run the deployment cell.


### 4.2. Deploy Model to Endpoint

This step deploys the trained model to a live endpoint, which makes it available to serve real-time predictions. This process can take 10-15 minutes.

In [21]:
# Define endpoint configuration and create it.
ENDPOINT_DISPLAY_NAME = "car_damage_endpoint"
endpoint = aiplatform.Endpoint.create(display_name=ENDPOINT_DISPLAY_name)

# Deploy the model to the newly created endpoint.
print("Deploying model to endpoint... (This may take 10-15 minutes)")
model.deploy(endpoint=endpoint)
print("Model deployed successfully.")

Creating a new endpoint...
Creating Endpoint
Create Endpoint backing LRO: projects/601447762723/locations/us-central1/endpoints/7476682367411683328/operations/2383582295228416000
Endpoint created. Resource name: projects/601447762723/locations/us-central1/endpoints/7476682367411683328
To use this Endpoint in another session:
endpoint = aiplatform.Endpoint('projects/601447762723/locations/us-central1/endpoints/7476682367411683328')
Deploying model to endpoint... (This may take 10-15 minutes)
Deploying model to Endpoint : projects/601447762723/locations/us-central1/endpoints/7476682367411683328
Deploy Endpoint model backing LRO: projects/601447762723/locations/us-central1/endpoints/7476682367411683328/operations/1626414607876751360
Endpoint model deployed. Resource name: projects/601447762723/locations/us-central1/endpoints/7476682367411683328
Model deployed successfully.


### 4.3. Make a Prediction

With the model deployed, we can now send a new, unseen image to the endpoint to get a real-time classification.

In [27]:
# Define the path to a local test image.
TEST_IMAGE_PATH = "C:/Users/redae/Jupyter/Projects/Car Damage Vision/Car Damage Severity Dataset/Validation/02-moderate/0001.JPEG"

print(f"Loading image from: {TEST_IMAGE_PATH}")
# Read and encode the image file into base64 format.
with open(TEST_IMAGE_PATH, "rb") as f:
    file_content = f.read()
encoded_content = base64.b64encode(file_content).decode('utf-8')
instance = [{"content": encoded_content}]

# Send the prediction request to the endpoint.
print("Sending prediction request...")
prediction = endpoint.predict(instances=instance)

# Print the prediction result.
print("\n--- Prediction Results ---")
print(prediction)

Loading image from: C:/Users/redae/Jupyter/Projects/Car Damage Vision/Car Damage Severity Dataset/Validation/02-moderate/0001.JPEG
Sending prediction request...

--- Prediction Results ---
Prediction(predictions=[{'confidences': [0.057875067, 0.899072111, 0.0430528224], 'ids': ['690410563993337856', '3572714325510455296', '8184400343937843200'], 'displayNames': ['01-minor', '02-moderate', '03-severe']}], deployed_model_id='1061612561478189056', metadata=None, model_version_id='1', model_resource_name='projects/601447762723/locations/us-central1/models/576094614931374080', explanations=None)


--- 
## 5. Project Cleanup

This is the final and most important step to ensure we stay within the free tier. This script finds and deletes all the cloud resources created during the project (Endpoint, Model, Dataset, and GCS Bucket).

In [31]:
# Define all necessary configuration variables.
PROJECT_ID = "reda-car-damage-vision"
LOCATION = "us-central1"
MODEL_ID = "576094614931374080"
ENDPOINT_DISPLAY_NAME = "car_damage_endpoint"
DATASET_DISPLAY_NAME = "car-damage-dataset"
BUCKET_NAME = f"car-damage-dataset-{PROJECT_ID}"

# Initialize the SDK.
aiplatform.init(project=PROJECT_ID, location=LOCATION)

# Find and delete all created resources.
print("Finding all project resources to delete...")

# Find and delete the endpoint.
endpoints = aiplatform.Endpoint.list(filter=f'display_name="{ENDPOINT_DISPLAY_NAME}"')
if endpoints:
    endpoint = endpoints[0]
    endpoint.undeploy_all()
    endpoint.delete()
    print("Endpoint deleted successfully.")
else:
    print("Endpoint not found or already deleted.")

# Find and delete the model by its ID.
try:
    model_resource_name = f"projects/{PROJECT_ID}/locations/{LOCATION}/models/{MODEL_ID}"
    model = aiplatform.Model(model_name=model_resource_name)
    model.delete()
    print("Model deleted successfully.")
except Exception:
    print("Model not found or already deleted.")

# Find and delete the dataset.
datasets = aiplatform.ImageDataset.list(filter=f"display_name='{DATASET_DISPLAY_NAME}'")
if datasets:
    dataset = datasets[0]
    dataset.delete()
    print("Dataset deleted successfully.")
else:
    print("Dataset not found or already deleted.")

# Find and delete the GCS bucket.
try:
    storage_client = storage.Client(project=PROJECT_ID)
    bucket = storage_client.get_bucket(BUCKET_NAME)
    bucket.delete(force=True)
    print("Cloud Storage bucket deleted successfully.")
except Exception:
    print("Bucket not found or already deleted.")

print("\nCleanup complete.")

Finding all project resources to delete...
Endpoint not found or already deleted.
Model deleted successfully.
Dataset deleted successfully.
Cloud Storage bucket deleted successfully.

Cleanup complete.
